# Aula 16 - Notebook: Circuitos Eulerianos e Inspeção de Manutenção na Manufatura

Neste notebook modelamos a malha física das passarelas de manutenção (grafo não-dirigido) e implementamos o **Algoritmo de Hierholzer** para gerar a rota ótima de inspeção do técnico ou robô industrial autônomo, garantindo que todas as esteiras sejam inspecionadas exatamente uma vez.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from collections import defaultdict
from typing import List, Tuple

class GrafoInspecaoEuleriano:
    def __init__(self):
        self.adj = defaultdict(list)
        
    def adicionar_passarela(self, u: str, v: str, id_passarela: str):
        self.adj[u].append((v, id_passarela))
        self.adj[v].append((u, id_passarela))
        
    def verificar_euleriano(self) -> Tuple[bool, List[str]]:
        impares = [v for v, viz in self.adj.items() if len(viz) % 2 != 0]
        return len(impares) == 0, impares

    def calcular_circuito_hierholzer(self, inicio: str) -> List[str]:
        adj_copia = {u: list(viz) for u, viz in self.adj.items()}
        pilha = [inicio]
        circuito = []
        while pilha:
            u = pilha[-1]
            if adj_copia[u]:
                v, id_e = adj_copia[u].pop()
                adj_copia[v].remove((u, id_e))
                pilha.append(v)
            else:
                circuito.append(pilha.pop())
        circuito.reverse()
        return circuito

# Modelagem da Planta de Manufatura: Construção do Grafo
g_insp = GrafoInspecaoEuleriano()

# Anel externo (Todas as estações conectadas sequencialmente)
g_insp.adicionar_passarela("Base_Manutencao", "Estacao_Silo", "p1")
g_insp.adicionar_passarela("Estacao_Silo", "Estacao_Inspecao", "p2")
g_insp.adicionar_passarela("Estacao_Inspecao", "Estacao_Triagem", "p3")
g_insp.adicionar_passarela("Estacao_Triagem", "Caixa_Lote_1", "p4")
g_insp.adicionar_passarela("Caixa_Lote_1", "Caixa_Lote_2", "p5")
g_insp.adicionar_passarela("Caixa_Lote_2", "Base_Manutencao", "p6")

# Anel interno balanceado (Criando caminhos transversais entre as estações críticas)
g_insp.adicionar_passarela("Estacao_Silo", "Estacao_Triagem", "p7")
g_insp.adicionar_passarela("Estacao_Triagem", "Estacao_Inspecao", "p8")
g_insp.adicionar_passarela("Estacao_Inspecao", "Estacao_Silo", "p9")

eul, imp = g_insp.verificar_euleriano()
print(f"Grafo é Euleriano: {eul} | Estações com Grau Ímpar: {imp}")

rota_tec = g_insp.calcular_circuito_hierholzer("Base_Manutencao")
print("Circuito Euleriano de Inspeção:", " -> ".join(rota_tec))
assert eul is True
